In [2]:
# load lib
library(tidyverse)
county_data <- read.csv("data/county_data.csv")
census_data <- read.csv("data/texas_census.csv")
brfss_data <- read.csv("data/brfss_data.csv")

Warning message:
"Paket 'forcats' wurde unter R Version 4.4.1 erstellt"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.1     v stringr   1.5.1
v ggplot2   3.5.1     v tibble    3.2.1
v lubridate 1.9.3     v tidyr     1.3.1
v purrr     1.0.2     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x tidyr::expand() masks Matrix::expand()
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
x tidyr::pack()   masks Matrix::pack()
x tidyr::unpack() masks Matrix::unpack()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
county_data <- county_data %>%
  left_join(census_data, by = "County")

# # Verify merge worked
county_data %>%
  summarise(
    n_counties        = n(),
    missing_hispanic  = sum(is.na(pct_hispanic)),
    missing_poverty   = sum(is.na(pct_poverty)),
    missing_uninsured = sum(is.na(pct_uninsured))
  )
nacounty <- county_data[!complete.cases(county_data), ]
print(nacounty)

n_counties,missing_hispanic,missing_poverty,missing_uninsured
<int>,<int>,<int>,<int>
254,4,4,4


       County  cve outbreak enrollment PHR pct_hispanic pct_black pct_white
62     DeWitt 2.14        0         NA   8           NA        NA        NA
151    Loving 0.00        0         NA   9            0         0         0
160 McCulloch 1.68        0         NA   9           NA        NA        NA
161  McLennan 1.82        9         NA   7           NA        NA        NA
162  McMullen 8.30        0         NA  11           NA        NA        NA
    pct_poverty pct_uninsured pct_college median_income pct_foreign_born
62           NA            NA          NA            NA               NA
151         6.1             0           0         51250                0
160          NA            NA          NA            NA               NA
161          NA            NA          NA            NA               NA
162          NA            NA          NA            NA               NA


In [4]:
county_data <- county_data %>%
  left_join(brfss_data, by = "PHR")

In [5]:
county_data %>% 
  select(County, PHR, cve, 
         medical_cost_pct, flu_shot_pct) %>% 
  head(10)


,County,PHR,cve,medical_cost_pct,flu_shot_pct
,<chr>,<int>,<dbl>,<dbl>,<dbl>
1,Anderson,4,2.54,21.6,25.4
2,Andrews,9,1.91,16.8,24.5
3,Angelina,5,2.50,18.1,25.6
4,Aransas,11,2.06,16.9,32.0
5,Archer,2,2.70,20.9,37.4
6,Armstrong,1,5.24,20.0,33.6
7,Atascosa,8,1.08,18.6,38.0
8,Austin,6,3.55,17.3,36.2
9,Bailey,1,0.80,20.0,33.6


In [11]:
library(lme4)
model_full <- glmer(
  outbreak ~
    # BRFSS PHR-level variables
    scale(medical_cost_pct) +
    scale(flu_shot_pct) +
    scale(cve) +
    # Census county-level proxies for suppressed BRFSS groups
    scale(pct_hispanic) +
    scale(pct_black) +
    scale(pct_poverty) +
    scale(pct_uninsured) +
    scale(pct_college) +
    
    # Offset
    offset(log(enrollment)) +
    
    # Random effects
    (1 | PHR) +
    (1 | PHR:County),
    
  family = poisson,
  data = county_data,
  nAGQ = 0
)
summary(model_full)

Generalized linear mixed model fit by maximum likelihood (Adaptive
  Gauss-Hermite Quadrature, nAGQ = 0) [glmerMod]
 Family: poisson  ( log )
Formula: 
outbreak ~ scale(medical_cost_pct) + scale(flu_shot_pct) + scale(cve) +  
    scale(pct_hispanic) + scale(pct_black) + scale(pct_poverty) +  
    scale(pct_uninsured) + scale(pct_college) + offset(log(enrollment)) +  
    (1 | PHR) + (1 | PHR:County)
   Data: county_data

      AIC       BIC    logLik -2*log(L)  df.resid 
    379.8     418.5    -178.9     357.8       238 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-0.60943 -0.27960 -0.14126 -0.04327  1.03649 

Random effects:
 Groups     Name        Variance Std.Dev.
 PHR:County (Intercept) 4.303    2.074   
 PHR        (Intercept) 5.153    2.270   
Number of obs: 249, groups:  PHR:County, 249; PHR, 11

Fixed effects:
                         Estimate Std. Error z value Pr(>|z|)    
(Intercept)             -11.60810    0.83266 -13.941   <2e-16 ***
scale(medical_co

In [26]:
model_measles <- glmer(
  outbreak ~
    scale(medical_cost_pct) +
    scale(flu_shot_pct) +
    scale(cve) +
    scale(pct_uninsured) +
    offset(log(enrollment)) +
    (1 | PHR),
    
  family = poisson,
  data = county_data,
  control = glmerControl(optimizer = "bobyqa",
                       optCtrl = list(maxfun = 1e6)) 
)
summary(model_measles)


Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: poisson  ( log )
Formula: 
outbreak ~ scale(medical_cost_pct) + scale(flu_shot_pct) + scale(cve) +  
    scale(pct_uninsured) + offset(log(enrollment)) + (1 | PHR)
   Data: county_data
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e+06))

      AIC       BIC    logLik -2*log(L)  df.resid 
    854.5     875.7    -421.3     842.5       243 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-6.0901 -0.4463 -0.1599 -0.0234 29.9704 

Random effects:
 Groups Name        Variance Std.Dev.
 PHR    (Intercept) 6.722    2.593   
Number of obs: 249, groups:  PHR, 11

Fixed effects:
                         Estimate Std. Error z value Pr(>|z|)    
(Intercept)             -12.06911    0.99183 -12.168  < 2e-16 ***
scale(medical_cost_pct)   2.92525    1.22777   2.383   0.0172 *  
scale(flu_shot_pct)      -0.28101    0.85878  -0.327   0.7435    
scale(cve)           

In [28]:
library(glmmTMB)

model_measles <- glmmTMB(
  outbreak ~ scale(medical_cost_pct) + scale(flu_shot_pct)  + scale(cve) + scale(pct_uninsured) +
    offset(log(enrollment)) +
    (1 | PHR),
  family = nbinom2,  # directly models overdispersion
  data = county_data
)
summary(model_measles)

 Family: nbinom2  ( log )
Formula:          
outbreak ~ scale(medical_cost_pct) + scale(flu_shot_pct) + scale(cve) +  
    scale(pct_uninsured) + offset(log(enrollment)) + (1 | PHR)
Data: county_data

      AIC       BIC    logLik -2*log(L)  df.resid 
    376.6     401.2    -181.3     362.6       242 

Random effects:

Conditional model:
 Groups Name        Variance Std.Dev.
 PHR    (Intercept) 4.218    2.054   
Number of obs: 249, groups:  PHR, 11

Dispersion parameter for nbinom2 family (): 0.159 

Conditional model:
                        Estimate Std. Error z value Pr(>|z|)    
(Intercept)             -11.0640     0.8737 -12.664  < 2e-16 ***
scale(medical_cost_pct)   2.1518     0.9903   2.173  0.02979 *  
scale(flu_shot_pct)      -0.1969     0.7193  -0.274  0.78422    
scale(cve)                0.7311     0.3573   2.046  0.04072 *  
scale(pct_uninsured)      0.7965     0.2854   2.791  0.00526 ** 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

In [8]:
summary(county_data$enrollment)
any(county_data$enrollment <= 0, na.rm = TRUE)

summary(county_data$cve)
any(!is.finite(county_data$cve))
any(county_data$cve < 0, na.rm = TRUE)
any(abs(county_data$cve - round(county_data$cve)) > 1e-8, na.rm = TRUE)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
     93    1180    3240   22023   10549  879002       5 

[1] FALSE

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   1.492   2.490   2.817   3.765  14.540 

[1] FALSE

[1] FALSE

[1] TRUE

In [9]:
library(lme4)

d <- subset(county_data, is.finite(cve))
d <- droplevels(d)

model_gauss <- lmer(
  cve ~
    scale(medical_cost_pct) +
    scale(flu_shot_pct) +
    scale(pct_hispanic) +
    scale(pct_black) +
    scale(pct_poverty) +
    scale(pct_uninsured) +
    scale(pct_college) +
    (1 | PHR),
  data = d,
  REML = TRUE
)

summary(model_gauss)

Linear mixed model fit by REML ['lmerMod']
Formula: 
cve ~ scale(medical_cost_pct) + scale(flu_shot_pct) + scale(pct_hispanic) +  
    scale(pct_black) + scale(pct_poverty) + scale(pct_uninsured) +  
    scale(pct_college) + (1 | PHR)
   Data: d

REML criterion at convergence: 1006.4

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.2546 -0.5531 -0.2283  0.5076  5.9887 

Random effects:
 Groups   Name        Variance Std.Dev.
 PHR      (Intercept) 0.4625   0.6801  
 Residual             2.9946   1.7305  
Number of obs: 250, groups:  PHR, 11

Fixed effects:
                        Estimate Std. Error t value
(Intercept)              2.71692    0.23750  11.440
scale(medical_cost_pct)  0.19122    0.26263   0.728
scale(flu_shot_pct)     -0.14831    0.24426  -0.607
scale(pct_hispanic)     -0.03826    0.11307  -0.338
scale(pct_black)        -0.24956    0.11701  -2.133
scale(pct_poverty)      -0.41578    0.13398  -3.103
scale(pct_uninsured)     0.18232    0.13036   1.399
scale(p